# Notebook 01 — Create Analytical Dataset

**Project:** Customer Review Prediction — Olist Brazilian E-Commerce  

Hey there! In this first notebook, I'm going to build our master analytical dataset. The goal here is to take all those raw, cleaned CSV files and combine them into a single, clean table where each row represents one order.

---

## What I'm Doing Here

1. First, I'll load up all the cleaned Olist CSV files.
2. Then, I'm going to join them all together (this basically does what `sql/analytical_queries.sql` would do).
3. I'll filter out the neutral reviews (score = 3) because they just add noise.
4. Next, I'll create our target variable: `positive_review`.
5. Finally, I'll save everything into `data/analytical/analytical_dataset.csv`.

## Why Python and not SQL?

I decided to do all the joins using `pandas.merge()`. This perfectly replicates the SQL logic, but it means I don't need a database server running. It makes the project super easy to run anywhere!

## 1. Imports
Let's bring in the libraries I'll need.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Display settings — show more columns and rows for exploration
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.4f}'.format)

print('Imports complete.')

Imports complete.


## 2. Constants and File Paths

I always like to use constants for file paths instead of hard-coding them everywhere. It makes my life so much easier if I ever need to change the folder structure later.

In [2]:
# ---------------------------------------------------------------------------
# Paths are relative to the notebook location (notebooks/).
# Going up two levels reaches the main project directory, then into cleaned/.
# ---------------------------------------------------------------------------
CLEANED_DATA_DIR = Path('../../cleaned')
ANALYTICAL_OUTPUT_DIR = Path('../data/analytical')

# ---------------------------------------------------------------------------
# Review score thresholds
# Positive: score >= 4 (clear satisfaction)
# Negative: score <= 2 (clear dissatisfaction)
# Neutral:  score == 3 (ambiguous; excluded from the binary classification)
# ---------------------------------------------------------------------------
POSITIVE_REVIEW_THRESHOLD = 4
NEUTRAL_SCORE = 3

# Ensure the output directory exists before saving
ANALYTICAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Cleaned data directory : {CLEANED_DATA_DIR.resolve()}')
print(f'Output directory       : {ANALYTICAL_OUTPUT_DIR.resolve()}')

Cleaned data directory : C:\Users\VishalReddyK\OneDrive\Documents\semester 2\statiscal machine learning\main project\cleaned
Output directory       : C:\Users\VishalReddyK\OneDrive\Documents\semester 2\statiscal machine learning\main project\customer_review_prediction\data\analytical


## 3. Load Cleaned Datasets

Now, I'm loading all the CSV files that were cleaned up previously. I'm assuming the data is already clean, so I won't do any extra cleaning here.

In [3]:
def load_csv(directory: Path, filename: str) -> pd.DataFrame:
    """
    Load a single CSV file from the specified directory.

    Parameters
    ----------
    directory : Path
        The directory containing the CSV file.
    filename : str
        The name of the CSV file (including the .csv extension).

    Returns
    -------
    pd.DataFrame
        The loaded DataFrame.

    Raises
    ------
    FileNotFoundError
        If the file does not exist at the expected path.
    """
    filepath = directory / filename
    if not filepath.exists():
        raise FileNotFoundError(f'Expected file not found: {filepath.resolve()}')
    dataframe = pd.read_csv(filepath)
    print(f'  Loaded {filename:<50}  shape={dataframe.shape}')
    return dataframe


print('Loading all cleaned datasets...\n')

orders_df               = load_csv(CLEANED_DATA_DIR, 'olist_orders_dataset.csv')
customers_df            = load_csv(CLEANED_DATA_DIR, 'olist_customers_dataset.csv')
order_items_df          = load_csv(CLEANED_DATA_DIR, 'olist_order_items_dataset.csv')
products_df             = load_csv(CLEANED_DATA_DIR, 'olist_products_dataset.csv')
payments_df             = load_csv(CLEANED_DATA_DIR, 'olist_order_payments_dataset.csv')
reviews_df              = load_csv(CLEANED_DATA_DIR, 'olist_order_reviews_dataset.csv')
sellers_df              = load_csv(CLEANED_DATA_DIR, 'olist_sellers_dataset.csv')
category_translation_df = load_csv(CLEANED_DATA_DIR, 'product_category_name_translation.csv')

print('\nAll datasets loaded successfully.')

Loading all cleaned datasets...



  Loaded olist_orders_dataset.csv                            shape=(99441, 8)
  Loaded olist_customers_dataset.csv                         shape=(99441, 5)


  Loaded olist_order_items_dataset.csv                       shape=(112650, 7)
  Loaded olist_products_dataset.csv                          shape=(32951, 9)
  Loaded olist_order_payments_dataset.csv                    shape=(103886, 5)


  Loaded olist_order_reviews_dataset.csv                     shape=(98673, 7)
  Loaded olist_sellers_dataset.csv                           shape=(3095, 4)
  Loaded product_category_name_translation.csv               shape=(74, 2)

All datasets loaded successfully.


## 4. Build the Analytical Dataset

Alright, time to assemble the master dataset. I'll do this step-by-step, just like how I'd write a big SQL query.

### Step 1 — Join Orders with Customer Info

The `orders` table has one row per order but only gives me a `customer_id` (which changes per order). I need to join it with `customers` to get the `customer_unique_id`. This is super important because I need to track the same real person across multiple orders later to build their history!

In [4]:
# Select only the customer columns we actually need to keep the dataset tidy
customer_columns_to_keep = ['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']

orders_with_customers = orders_df.merge(
    customers_df[customer_columns_to_keep],
    on='customer_id',
    how='inner'   # inner join: we only want orders where customer info exists
)

print(f'Orders before joining with customers : {len(orders_df):>7,}')
print(f'Orders after joining with customers  : {len(orders_with_customers):>7,}')

Orders before joining with customers :  99,441
Orders after joining with customers  :  99,441


### Step 2 — Filter Reviews and Create the Binary Target Variable

I'm dropping any orders with a 3-star review. Why? Because a 3-star review is basically "meh" — they aren't happy, but they aren't angry either. If I leave them in, it just confuses the model.

| Score | Meaning | Target |
|-------|---------|--------|
| 1, 2  | Negative | 0 |
| 3     | Neutral  | **dropped** |
| 4, 5  | Positive | 1 |

In [5]:
# Keep only reviews with a clear positive or negative score
reviews_binary = reviews_df[reviews_df['review_score'] != NEUTRAL_SCORE].copy()

# Create the binary target variable
# True (1) = positive review (score 4 or 5)
# False (0) = negative review (score 1 or 2)
reviews_binary['positive_review'] = (
    reviews_binary['review_score'] >= POSITIVE_REVIEW_THRESHOLD
).astype(int)

print(f'Total reviews in dataset           : {len(reviews_df):>7,}')
print(f'Reviews with neutral score (=3)    : {(reviews_df["review_score"] == NEUTRAL_SCORE).sum():>7,}')
print(f'Reviews retained for modelling     : {len(reviews_binary):>7,}')
print()
print('Target variable distribution:')
target_counts = reviews_binary['positive_review'].value_counts().sort_index()
for label_value, count in target_counts.items():
    label_name = 'Positive (1)' if label_value == 1 else 'Negative (0)'
    percentage = count / len(reviews_binary) * 100
    print(f'  {label_name}: {count:>7,}  ({percentage:.1f}%)')

# Keep only the columns needed from reviews
review_columns_to_keep = ['order_id', 'review_score', 'positive_review']

# Join orders+customers with the filtered reviews
# inner join: we only want orders that have a binary review
orders_with_reviews = orders_with_customers.merge(
    reviews_binary[review_columns_to_keep],
    on='order_id',
    how='inner'
)

print(f'\nOrders with binary review labels: {len(orders_with_reviews):>7,}')

Total reviews in dataset           :  98,673
Reviews with neutral score (=3)    :   8,133
Reviews retained for modelling     :  90,540

Target variable distribution:
  Negative (0):  14,495  (16.0%)
  Positive (1):  76,045  (84.0%)

Orders with binary review labels:  90,540


### Step 3 — Prepare Products with English Category Names

The original product categories are in Portuguese, so I'll join the translation table to get the English names. I'm also going to quickly calculate the product volume (just a simple box approximation) so I can add it up for the whole order later.

In [6]:
# Join products with category translation
products_with_categories = products_df.merge(
    category_translation_df,
    on='product_category_name',
    how='left'   # left join: keep all products, even those without a translation
)

# Where an English translation is missing, fall back to the Portuguese name,
# and if that is also missing, use 'unknown'
products_with_categories['product_category_name_english'] = (
    products_with_categories['product_category_name_english']
    .fillna(products_with_categories['product_category_name'])
    .fillna('unknown')
)

# Compute product volume as a rectangular box (length × height × width)
# This is a reasonable physical approximation for packaging size
products_with_categories['product_volume_cm3'] = (
    products_with_categories['product_length_cm']
    * products_with_categories['product_height_cm']
    * products_with_categories['product_width_cm']
)

print(f'Products loaded               : {len(products_df):>7,}')
print(f'Products with English name    : {products_with_categories["product_category_name_english"].notna().sum():>7,}')
print(f'Unique English categories     : {products_with_categories["product_category_name_english"].nunique():>7,}')

Products loaded               :  32,951
Products with English name    :  32,951
Unique English categories     :      74


### Step 4 — Aggregate Order Items

Since one order can have multiple items (maybe even from different sellers), I need to roll them all up into a single row per order. I'm calculating things like:

- **total_price** — how much the items cost in total
- **total_freight_value** — the total shipping cost
- **number_of_items** — how many things they bought
- **seller_count** — how many different sellers are involved
- **total_product_weight_g** — total weight
- **total_product_volume_cm3** — total volume
- **number_of_categories** — how many different types of products are in the box

In [7]:
# First, join order items with product information (weight, volume, category)
product_columns_needed = [
    'product_id',
    'product_category_name_english',
    'product_weight_g',
    'product_volume_cm3'
]

order_items_with_products = order_items_df.merge(
    products_with_categories[product_columns_needed],
    on='product_id',
    how='left'   # left join: keep all order items, even if product details are missing
)

# Aggregate all item-level information to the order level
order_item_aggregates = (
    order_items_with_products
    .groupby('order_id')
    .agg(
        total_price=('price', 'sum'),
        total_freight_value=('freight_value', 'sum'),
        number_of_items=('order_item_id', 'count'),
        seller_count=('seller_id', 'nunique'),
        total_product_weight_g=('product_weight_g', 'sum'),
        total_product_volume_cm3=('product_volume_cm3', 'sum'),
        number_of_categories=('product_category_name_english', 'nunique')
    )
    .reset_index()
)

print(f'Order item aggregates computed for {len(order_item_aggregates):,} orders.')
print()
print(order_item_aggregates.head(3))

Order item aggregates computed for 98,666 orders.

                           order_id  total_price  total_freight_value  \
0  00010242fe8c5a6d1ba2dd792cb16214      58.9000              13.2900   
1  00018f77f2f0320c557190d7a144bdd3     239.9000              19.9300   
2  000229ec398224ef6ca0657da4fc703e     199.0000              17.8700   

   number_of_items  seller_count  total_product_weight_g  \
0                1             1                650.0000   
1                1             1             30,000.0000   
2                1             1              3,050.0000   

   total_product_volume_cm3  number_of_categories  
0                3,528.0000                     1  
1               60,000.0000                     1  
2               14,157.0000                     1  


### Step 5 — Compute the Dominant Product Category per Order

If someone buys a TV and a cheap HDMI cable in the same order, their experience is mostly going to be about the TV. So, I'm finding the "dominant category" for each order based on which category costs the most. This is basically the Python version of a SQL `ROW_NUMBER() OVER (PARTITION BY...)` trick.

In [8]:
def compute_dominant_category(items_with_products_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute the dominant product category for each order.

    The dominant category is defined as the product category that contributes
    the highest total price (sum of item prices) within an order.

    Parameters
    ----------
    items_with_products_df : pd.DataFrame
        DataFrame containing order_id, product_category_name_english, and price.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns [order_id, dominant_product_category].
    """
    # Sum revenue by (order, category)
    category_revenue_by_order = (
        items_with_products_df
        .groupby(['order_id', 'product_category_name_english'])['price']
        .sum()
        .reset_index()
        .rename(columns={'price': 'category_revenue'})
    )

    # Sort descending by revenue, then take the top category for each order
    dominant_category_per_order = (
        category_revenue_by_order
        .sort_values('category_revenue', ascending=False)
        .groupby('order_id')
        .first()
        .reset_index()
        [['order_id', 'product_category_name_english']]
        .rename(columns={'product_category_name_english': 'dominant_product_category'})
    )

    return dominant_category_per_order


dominant_category_per_order = compute_dominant_category(order_items_with_products)

print(f'Dominant category computed for {len(dominant_category_per_order):,} orders.')
print()
print('Top 10 dominant product categories:')
print(dominant_category_per_order['dominant_product_category'].value_counts().head(10))

Dominant category computed for 98,666 orders.

Top 10 dominant product categories:
dominant_product_category
bed_bath_table           9342
health_beauty            8801
sports_leisure           7682
computers_accessories    6671
furniture_decor          6335
housewares               5821
watches_gifts            5606
telephony                4179
auto                     3878
toys                     3871
Name: count, dtype: int64


### Step 6 — Identify the Primary Seller per Order

Just like with categories, an order might involve multiple sellers. I'm tagging the "primary seller" as the one who made the most money off the order. I'll use this seller's location later to figure out if the package is shipping within the same state.

In [9]:
def identify_primary_seller(items_df: pd.DataFrame, sellers_df: pd.DataFrame) -> pd.DataFrame:
    """
    Identify the primary seller for each order.

    The primary seller is defined as the seller who contributed the highest
    total item price (sum of prices for their items in that order).

    Parameters
    ----------
    items_df : pd.DataFrame
        Order items DataFrame containing order_id, seller_id, and price.
    sellers_df : pd.DataFrame
        Sellers DataFrame containing seller_id and seller_state.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns [order_id, primary_seller_id, seller_state].
    """
    # Compute total revenue per (order, seller)
    seller_revenue_by_order = (
        items_df
        .groupby(['order_id', 'seller_id'])['price']
        .sum()
        .reset_index()
        .rename(columns={'price': 'seller_revenue'})
    )

    # Select the seller with the highest revenue for each order
    primary_seller_per_order = (
        seller_revenue_by_order
        .sort_values('seller_revenue', ascending=False)
        .groupby('order_id')
        .first()
        .reset_index()
        [['order_id', 'seller_id']]
        .rename(columns={'seller_id': 'primary_seller_id'})
    )

    # Attach the seller's state
    primary_seller_with_state = primary_seller_per_order.merge(
        sellers_df[['seller_id', 'seller_state']],
        left_on='primary_seller_id',
        right_on='seller_id',
        how='left'
    ).drop(columns=['seller_id'])

    return primary_seller_with_state


primary_seller_info = identify_primary_seller(order_items_df, sellers_df)

print(f'Primary seller identified for {len(primary_seller_info):,} orders.')
print()
print('Top 5 seller states:')
print(primary_seller_info['seller_state'].value_counts().head(5))

Primary seller identified for 98,666 orders.



Top 5 seller states:
seller_state
SP    69944
MG     7858
PR     7602
RJ     4306
SC     3623
Name: count, dtype: int64


### Step 7 — Aggregate Payment Information

Sometimes people split payments (like using a voucher and a credit card). I'll just pick the "primary payment type" as the method they used to pay the largest chunk of the bill.

In [10]:
def aggregate_payments(payments_df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate payment records to one row per order.

    The primary payment type is the payment method with the highest payment value.
    The installments count is taken from that same primary payment record.

    Parameters
    ----------
    payments_df : pd.DataFrame
        Payments DataFrame with order_id, payment_type, payment_installments,
        and payment_value.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns [order_id, payment_type, payment_installments].
    """
    # Sort so that the highest-value payment appears first for each order
    payments_sorted = payments_df.sort_values('payment_value', ascending=False)

    # Take the first record for each order (highest-value payment)
    primary_payment = (
        payments_sorted
        .groupby('order_id')
        .first()
        .reset_index()
        [['order_id', 'payment_type', 'payment_installments']]
    )

    return primary_payment


payment_info = aggregate_payments(payments_df)

print(f'Payment info aggregated for {len(payment_info):,} orders.')
print()
print('Payment type distribution:')
print(payment_info['payment_type'].value_counts())

Payment info aggregated for 99,440 orders.

Payment type distribution:
payment_type
credit_card    74975
boleto         19784
voucher         3151
debit_card      1527
not_defined        3
Name: count, dtype: int64


### Step 8 — Assemble the Final Analytical Dataset

Time to bring it all together! I'll take my base `orders_with_reviews` table and left-join all the pieces I just built. Using left joins makes sure I don't lose any orders, even if they're missing some secondary details.

In [11]:
# Assemble the analytical dataset by merging one component at a time.
# Using individual merge steps (rather than chaining) makes each join explicit
# and easier to debug if a problem arises.

analytical_dataset = orders_with_reviews.copy()

# --- Add order item aggregates (price, freight, counts) ---
analytical_dataset = analytical_dataset.merge(
    order_item_aggregates,
    on='order_id',
    how='left'
)

# --- Add dominant product category ---
analytical_dataset = analytical_dataset.merge(
    dominant_category_per_order,
    on='order_id',
    how='left'
)

# --- Add primary seller info (seller_id and seller_state) ---
analytical_dataset = analytical_dataset.merge(
    primary_seller_info,
    on='order_id',
    how='left'
)

# --- Add payment information ---
analytical_dataset = analytical_dataset.merge(
    payment_info,
    on='order_id',
    how='left'
)

print(f'Final analytical dataset shape: {analytical_dataset.shape}')
print(f'\nColumns ({len(analytical_dataset.columns)}):')
for col in analytical_dataset.columns:
    print(f'  {col}')

Final analytical dataset shape: (90540, 25)

Columns (25):
  order_id
  customer_id
  order_status
  order_purchase_timestamp
  order_approved_at
  order_delivered_carrier_date
  order_delivered_customer_date
  order_estimated_delivery_date
  customer_unique_id
  customer_state
  customer_city
  review_score
  positive_review
  total_price
  total_freight_value
  number_of_items
  seller_count
  total_product_weight_g
  total_product_volume_cm3
  number_of_categories
  dominant_product_category
  primary_seller_id
  seller_state
  payment_type
  payment_installments


## 5. Data Quality Check

Before I save this, let's do a quick sanity check:
- Make sure I didn't accidentally create duplicate orders during the joins.
- Check for missing values.
- See what my positive/negative review balance looks like.

In [12]:
print('=' * 55)
print('   DATA QUALITY REPORT')
print('=' * 55)

# Check for duplicate order IDs
duplicate_orders = analytical_dataset.duplicated(subset='order_id').sum()
print(f'\nDuplicate order_id rows : {duplicate_orders}')

# Missing values summary
missing_counts = analytical_dataset.isnull().sum()
missing_percentages = (missing_counts / len(analytical_dataset) * 100).round(2)
missing_report = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct': missing_percentages
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

print(f'\nMissing Values:')
if missing_report.empty:
    print('  None found.')
else:
    print(missing_report.to_string())

# Target class balance
print(f'\nTarget Variable Distribution (positive_review):')
target_counts = analytical_dataset['positive_review'].value_counts().sort_index()
for label_value, count in target_counts.items():
    label_name = 'Positive (score 4-5)' if label_value == 1 else 'Negative (score 1-2)'
    percentage = count / len(analytical_dataset) * 100
    print(f'  {label_value}  {label_name}: {count:>7,}  ({percentage:.1f}%)')

# Dataset size
memory_mb = analytical_dataset.memory_usage(deep=True).sum() / 1024 ** 2
print(f'\nDataset shape  : {analytical_dataset.shape}')
print(f'Memory usage   : {memory_mb:.2f} MB')
print('=' * 55)

   DATA QUALITY REPORT

Duplicate order_id rows : 0

Missing Values:
                               missing_count  missing_pct
order_delivered_customer_date           2627       2.9000
order_delivered_carrier_date            1643       1.8100
total_price                              703       0.7800
number_of_items                          703       0.7800
total_freight_value                      703       0.7800
seller_count                             703       0.7800
total_product_weight_g                   703       0.7800
primary_seller_id                        703       0.7800
total_product_volume_cm3                 703       0.7800
number_of_categories                     703       0.7800
dominant_product_category                703       0.7800
seller_state                             703       0.7800
order_approved_at                        137       0.1500
payment_type                               1       0.0000
payment_installments                       1       0.0000

Ta


Dataset shape  : (90540, 25)
Memory usage   : 40.54 MB


## 6. Preview the Analytical Dataset
Let's take a quick look at what I just built!

In [13]:
analytical_dataset.head(5)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_state,customer_city,review_score,positive_review,total_price,total_freight_value,number_of_items,seller_count,total_product_weight_g,total_product_volume_cm3,number_of_categories,dominant_product_category,primary_seller_id,seller_state,payment_type,payment_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,SP,sao paulo,4,1,29.9900,8.7200,1.0000,1.0000,500.0000,"1,976.0000",1.0000,housewares,3504c0cb71d7fa48d967e0e4c94d59d9,SP,voucher,1.0000
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,BA,barreiras,4,1,118.7000,22.7600,1.0000,1.0000,400.0000,"4,693.0000",1.0000,perfumery,289cdb325fb7e7f891c38608bf9e0962,SP,boleto,1.0000
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,GO,vianopolis,5,1,159.9000,19.2200,1.0000,1.0000,420.0000,"9,576.0000",1.0000,auto,4869f7a5dfa277a7dca6462dcf3b52b2,SP,credit_card,3.0000
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,RN,sao goncalo do amarante,5,1,45.0000,27.2000,1.0000,1.0000,450.0000,"6,000.0000",1.0000,pet_shop,66922902710d126a0e7d26b0e3805106,MG,credit_card,1.0000
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,SP,santo andre,5,1,19.9000,8.7200,1.0000,1.0000,250.0000,"11,475.0000",1.0000,stationery,2c9e548be18521d1c43cde1c582c6de8,SP,credit_card,1.0000


## 7. Save the Analytical Dataset

Looks good! I'm saving this out to `data/analytical/analytical_dataset.csv`. I'll be using this file as the starting point for all the other notebooks.

In [14]:
output_filepath = ANALYTICAL_OUTPUT_DIR / 'analytical_dataset.csv'
analytical_dataset.to_csv(output_filepath, index=False)

print(f'Analytical dataset saved to:')
print(f'  {output_filepath.resolve()}')
print(f'\nRows saved : {len(analytical_dataset):,}')
print(f'Columns    : {len(analytical_dataset.columns)}')
print('\nNotebook 01 complete. Proceed to notebooks/02_feature_engineering.ipynb')

Analytical dataset saved to:
  C:\Users\VishalReddyK\OneDrive\Documents\semester 2\statiscal machine learning\main project\customer_review_prediction\data\analytical\analytical_dataset.csv

Rows saved : 90,540
Columns    : 25

Notebook 01 complete. Proceed to notebooks/02_feature_engineering.ipynb
